# Lorentzian Model Definition

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from lmfit import minimize, Parameters, report_fit
import csv
import time

In [2]:
def lorentzian(x, x0, gamma, maximum):
    return maximum / (1.0 + ((x - x0) / gamma)**2)


def addWhiteNoiseRandom(yData, sigma):
    y_noisy = yData + np.random.normal(0, sigma, len(yData))
    return y_noisy


def addNoiseSNR(yData, SNR):
    signal_power = np.sum(yData**2) / len(yData)
    noise_power = np.sqrt(signal_power / SNR)
    y_noisy = yData + np.random.normal(0, noise_power, len(yData))
    return y_noisy, noise_power

In [3]:
def residual(params, x, data):
    x0 = params['x0']
    gamma = params['gamma']
    maximum = params['maximum']
    
    model = lorentzian(x, x0, gamma, maximum)
    return model - data

# Experiment I: Fit Success Rate per Noise per Algorithm & ROC curves

# Error Thresholds

In [ ]:
def center_pixel_error(true_val, fit_val, x_min, x_max, num_points, pixel_threshold):
    pixel_val = (x_max - x_min)/num_points
    deviation = abs(true_val - fit_val)
    pixel_deviation = deviation/pixel_val

    return pixel_deviation <= pixel_threshold
    

In [ ]:
def relative_error(true_val, fit_val, threshold):
    rel_deviation = abs(true_val - fit_val)/true_val
    return rel_deviation <= threshold

## Running the Experiment

In [20]:
#------------- Peak Params ------------#

num_points = 1000
x_min = -500.0
x_max = 500.0
center = 0.0
width = 5.0
amplitude = 1.0

#--------- Experiment Params ---------#

min_noise = 0.0      # as sigma values
max_noise = 0.5
num_noise_groups = 10 # How many groups of noise value x will there be

noise_levels = np.linspace(min_noise, max_noise, num_noise_groups)

samples_per_group = 1 # Number of spectra per noise group

#------------------------------------#

x_data = np.linspace(x_min, x_max, num_points)
y_clean = lorentzian(x_data, center, width, amplitude)

with open('Experiment_I_Results.csv', 'w', newline='') as savefile:
    writer = csv.writer(savefile, delimiter=',')
    writer.writerow(['id', 'noise_gp', 'noise_val', 'x0', 'gamma', 'amp', 'x0_f', 'gamma_f', 'amp_f', 'nfe', 'time (ms)'])

    index = 0
    noise_group_id = 0
    for noise in noise_levels:
        for i in range(samples_per_group):
            y_noisy = addWhiteNoiseRandom(y_clean, sigma=noise)
            
            #---- Initial Parameter Guesses ----#
            """
            In this first round of tests we use the true parameters
            as starting parameters.
            """
            
            params = Parameters()
            params.add('x0', value=center)
            params.add('gamma', value=width)
            params.add('maximum', value=amplitude)
            
            #------- Minimizer -------#

            start = time.perf_counter()
            result = minimize(residual, params, args=(x_data, y_noisy))
            end = time.perf_counter()

            #------- Results Saving -------#

            x0_f = result.params['x0'].value
            gamma_f = result.params['gamma'].value
            amp_f = result.params['maximum'].value
            
            nfe = result.nfev
            runtime = (end - start)*1000

            line = [index, noise_group_id, noise, center, width, amplitude, x0_f, gamma_f, amp_f, nfe, runtime]
            line = map(lambda x : str(x), line)
            
            writer.writerow(line)

            index += 1
        noise_group_id += 1
        

## Analysis of Results